# Nexus Natural Hashing v1

## Fixed-Width Handles for Infinite-Depth Fields

This notebook formalizes the reset:

$$
\boxed{
\text{hashing is not invented; hashing is how finite systems survive contact with infinite detail.}
}
$$

Natural hashing:

$$
\boxed{
\text{deep variable reality}
\rightarrow
\text{bounded readable handle}
\rightarrow
\text{action}
}
$$

The core claim is not that nature literally runs SHA-256.  
The claim is structural:

$$
\boxed{
\text{finite agents require bounded interfaces to act on unbounded fields.}
}
$$

So perception, objects, DNA, matter, memory, GUI rows, and cryptographic digests all share the same operational grammar:

$$
\Delta:\text{world changes}
$$

$$
\oplus:\text{organism/field couples to change}
$$

$$
\circlearrowleft:\text{memory/feedback updates pressure}
$$

$$
\bot:\text{stable glyph appears}
$$

$$
\Psi:\text{action proceeds on the glyph, not the full substrate}
$$

Executable goals:

1. Show that a deep state can be compressed into a fixed-width handle.
2. Show that handles preserve action-relevant structure while discarding irrelevant detail.
3. Show that many deep states can share one handle without destroying operational information.
4. Show that collision is not automatically failure; collision is failure only if the collapsed states require different actions.
5. Show that residue geometry is the fiber/nullspace behind a handle.
6. Bridge the result back to SHA: the digest is a fixed-width residue surface, but SHA is tuned for avalanche rather than semantic locality.

Scope guard:

$$
\boxed{
\text{This notebook models natural hashing as bounded interface formation. It does not claim SHA preimage recovery.}
}
$$


In [ ]:
# Imports and settings

import hashlib
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260504
random.seed(SEED)
np.random.seed(SEED)

N_STATES = 5000
DEEP_DIM = 256
HANDLE_BITS = 16
ACTION_DIM = 8

print("Natural Hashing notebook initialized.")
print(f"N_STATES={N_STATES}, DEEP_DIM={DEEP_DIM}, HANDLE_BITS={HANDLE_BITS}, ACTION_DIM={ACTION_DIM}")


## 1. Deep state versus handle

Let a deep state be:

$$
x\in\{0,1\}^{256}.
$$

A finite handle is:

$$
h=\Pi(x)\in\{0,1\}^{16}.
$$

The handle is not the full object. It is the bounded callable interface.

The action rule is another projection:

$$
a=A(x)\in\{0,1\}^{8}.
$$

A handle is operationally good when it preserves the action-relevant structure:

$$
h(x)=h(y)\Rightarrow A(x)\approx A(y).
$$

Collision is only harmful when:

$$
h(x)=h(y)
\quad\text{but}\quad
A(x)\ne A(y).
$$


In [ ]:
# Generate a synthetic deep field

# Deep states: random binary field vectors.
X = np.random.randint(0, 2, size=(N_STATES, DEEP_DIM), dtype=np.uint8)

# Action-relevant hidden structure: only a subset of dimensions matter.
# This models an organism that only needs action-relevant detail, not total detail.
action_indices = np.random.choice(DEEP_DIM, size=ACTION_DIM * 5, replace=False)
action_matrix = np.random.randint(0, 2, size=(ACTION_DIM, len(action_indices)), dtype=np.uint8)

def gf2_matmul(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    return (A @ B) & 1

def action_of_state(Xbatch: np.ndarray) -> np.ndarray:
    sub = Xbatch[:, action_indices]
    return (sub @ action_matrix.T) & 1

A_true = action_of_state(X)

print("Deep field:", X.shape)
print("Action readout:", A_true.shape)
print("Action-relevant dimensions:", len(action_indices), "/", DEEP_DIM)


## 2. Three handle types

We compare three ways to form fixed-width handles.

### Random projection handle

$$
h_R(x)=R x\pmod 2
$$

This is a raw bounded projection.

### Action-aligned handle

$$
h_A(x)=A(x)\ ||\ \eta(x)
$$

This preserves action variables first, then fills remaining bits with nuisance structure.

### Cryptographic handle

$$
h_C(x)=\operatorname{SHA256}(x)_{0:16}
$$

This is tuned for avalanche, not semantic locality.

These represent three interface philosophies:

$$
\boxed{
\text{measurement}
\quad
\text{need-aligned perception}
\quad
\text{cryptographic residue}
}
$$


In [ ]:
# Handle functions

random_projection = np.random.randint(0, 2, size=(HANDLE_BITS, DEEP_DIM), dtype=np.uint8)
nuisance_projection = np.random.randint(0, 2, size=(HANDLE_BITS - ACTION_DIM, DEEP_DIM), dtype=np.uint8)

def bits_to_int_le(bits: np.ndarray) -> int:
    out = 0
    for i, b in enumerate(bits):
        out |= (int(b) & 1) << i
    return out

def handle_random_projection(Xbatch: np.ndarray) -> np.ndarray:
    H = (Xbatch @ random_projection.T) & 1
    return H.astype(np.uint8)

def handle_action_aligned(Xbatch: np.ndarray) -> np.ndarray:
    action_bits = action_of_state(Xbatch)
    nuisance = (Xbatch @ nuisance_projection.T) & 1
    return np.concatenate([action_bits, nuisance], axis=1).astype(np.uint8)

def handle_sha_prefix(Xbatch: np.ndarray) -> np.ndarray:
    rows = []
    for x in Xbatch:
        packed = np.packbits(x)
        digest = hashlib.sha256(bytes(packed)).digest()
        bits = np.unpackbits(np.frombuffer(digest, dtype=np.uint8))[:HANDLE_BITS]
        rows.append(bits)
    return np.array(rows, dtype=np.uint8)

def handle_ids(H: np.ndarray) -> np.ndarray:
    return np.array([bits_to_int_le(row) for row in H], dtype=np.int64)

H_random = handle_random_projection(X)
H_action = handle_action_aligned(X)
H_sha = handle_sha_prefix(X)

handle_tables = {
    "random_projection": H_random,
    "action_aligned": H_action,
    "sha_prefix": H_sha,
}

print("Handles built.")
for name, H in handle_tables.items():
    print(name, H.shape, "unique handles:", len(np.unique(handle_ids(H))))


## 3. Collision is not automatically failure

A fixed-width handle must collide when the deep field is larger than the handle space.

But a collision is operationally safe if the colliding states require the same action.

For each handle type we compute:

$$
P(\text{same action}\mid \text{same handle})
$$

and:

$$
P(\text{action conflict}\mid \text{same handle}).
$$


In [ ]:
# Collision and action-coherence analysis

def action_id(Arow: np.ndarray) -> int:
    return bits_to_int_le(Arow)

action_ids = np.array([action_id(row) for row in A_true], dtype=np.int64)

def collision_report(H: np.ndarray, name: str) -> Dict[str, float]:
    hids = handle_ids(H)
    df = pd.DataFrame({"handle": hids, "action": action_ids})
    grouped = df.groupby("handle")
    collision_groups = [g for _, g in grouped if len(g) > 1]
    total_collision_states = sum(len(g) for g in collision_groups)
    conflicting_groups = 0
    conflict_states = 0
    coherent_groups = 0
    coherent_states = 0

    for g in collision_groups:
        n_actions = g["action"].nunique()
        if n_actions == 1:
            coherent_groups += 1
            coherent_states += len(g)
        else:
            conflicting_groups += 1
            conflict_states += len(g)

    return {
        "handle_type": name,
        "unique_handles": df["handle"].nunique(),
        "handle_space": 2 ** H.shape[1],
        "collision_groups": len(collision_groups),
        "collision_states": total_collision_states,
        "coherent_collision_groups": coherent_groups,
        "conflicting_collision_groups": conflicting_groups,
        "coherent_collision_state_rate": coherent_states / max(1, total_collision_states),
        "conflict_collision_state_rate": conflict_states / max(1, total_collision_states),
        "global_action_classes": df["action"].nunique(),
    }

reports = [collision_report(H, name) for name, H in handle_tables.items()]
collision_df = pd.DataFrame(reports)
collision_df


In [ ]:
# Plot collision conflict rates

plt.figure(figsize=(8, 4))
plt.bar(collision_df["handle_type"], collision_df["conflict_collision_state_rate"])
plt.title("Collision is harmful only when action differs")
plt.ylabel("conflict rate among collided states")
plt.xlabel("handle type")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 4. Handle as GUI row

A GUI row does not contain the whole file.

It contains enough information to act:

$$
\text{name},\quad \text{type},\quad \text{icon},\quad \text{size},\quad \text{modified time}.
$$

The deep file remains underneath.

Likewise, an organismic handle does not contain the whole world state. It contains enough to decide:

$$
\boxed{
\text{eat},\quad \text{avoid},\quad \text{approach},\quad \text{ignore},\quad \text{repair}
}
$$

This cell constructs a small “natural file list”: handles, action rows, and hidden deep-state multiplicities.


In [ ]:
# Natural GUI rows from action-aligned handle

hids = handle_ids(H_action)
df_gui = pd.DataFrame({
    "handle": hids,
    "action": action_ids,
})
summary_gui = (
    df_gui
    .groupby("handle")
    .agg(
        deep_states=("handle", "size"),
        action_classes=("action", "nunique"),
        primary_action=("action", lambda s: int(s.mode().iloc[0])),
    )
    .reset_index()
    .sort_values(["deep_states", "action_classes"], ascending=[False, False])
)

summary_gui["row_status"] = np.where(summary_gui["action_classes"] == 1, "stable glyph", "ambiguous glyph")
summary_gui.head(15)


## 5. Residue geometry: the fiber behind a handle

For a linear handle:

$$
h=Rx\pmod2
$$

the set of all deep states sharing a handle is an affine fiber:

$$
\{x:\ Rx=h\}.
$$

The hidden degrees of freedom are:

$$
\dim\ker(R)=n-\operatorname{rank}(R).
$$

This is the geometric residue behind the interface.

The handle does not destroy the fiber.  
It collapses it into a callable row.


In [ ]:
# GF(2) rank/nullity of handle projections

def gf2_rank(A: np.ndarray) -> int:
    A = (A.copy() & 1).astype(np.uint8)
    m, n = A.shape
    rank = 0
    row = 0
    for col in range(n):
        pivot = None
        for r in range(row, m):
            if A[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != row:
            A[[row, pivot]] = A[[pivot, row]]
        for r in range(m):
            if r != row and A[r, col]:
                A[r] ^= A[row]
        rank += 1
        row += 1
        if row == m:
            break
    return rank

rank_random_projection = gf2_rank(random_projection)
rank_action_projection = gf2_rank(np.vstack([
    np.pad(action_matrix, ((0,0),(0,DEEP_DIM-len(action_indices))), mode="constant")
]) if False else random_projection)  # placeholder not used

fiber_df = pd.DataFrame([
    {
        "projection": "random_projection",
        "input_dim": DEEP_DIM,
        "handle_bits": HANDLE_BITS,
        "rank": rank_random_projection,
        "fiber_dimension": DEEP_DIM - rank_random_projection,
        "states_per_full_fiber_if_linear": f"2^{DEEP_DIM - rank_random_projection}",
    }
])

fiber_df


## 6. Natural hashing versus cryptographic hashing

Natural hashing is often **need-aligned**:

$$
\Pi_{\text{body}}(W,H_{\text{memory}},N_{\text{need}})
$$

It preserves what matters for action.

Cryptographic hashing is deliberately **avalanche-aligned**:

$$
d_H(H(x),H(x\oplus e_i))\approx \frac{m}{2}.
$$

So SHA is a technical hash, but not a perception hash.

This cell compares one-bit sensitivity of the three handle types.


In [ ]:
# One-bit sensitivity of handles

def flip_bit_batch(Xbatch: np.ndarray, bit_index: int) -> np.ndarray:
    Y = Xbatch.copy()
    Y[:, bit_index] ^= 1
    return Y

sample_idx = np.random.choice(N_STATES, size=256, replace=False)
X_sample = X[sample_idx]
bits_to_probe = np.random.choice(DEEP_DIM, size=64, replace=False)

sens_rows = []
for bit in bits_to_probe:
    Y = flip_bit_batch(X_sample, int(bit))
    for name, func in [
        ("random_projection", handle_random_projection),
        ("action_aligned", handle_action_aligned),
        ("sha_prefix", handle_sha_prefix),
    ]:
        H0 = func(X_sample)
        H1 = func(Y)
        hd = (H0 ^ H1).sum(axis=1)
        sens_rows.append({
            "handle_type": name,
            "flipped_deep_bit": int(bit),
            "mean_handle_hd": float(hd.mean()),
            "std_handle_hd": float(hd.std()),
        })

sens_df = pd.DataFrame(sens_rows)
sens_summary = sens_df.groupby("handle_type")["mean_handle_hd"].describe().round(3)
sens_summary


In [ ]:
# Plot handle sensitivity

plt.figure(figsize=(8, 4))
sens_df.groupby("handle_type")["mean_handle_hd"].mean().sort_values().plot(kind="barh")
plt.title("Mean handle change from one deep-bit flip")
plt.xlabel("mean Hamming distance in handle")
plt.ylabel("handle type")
plt.tight_layout()
plt.show()


## 7. Perception as need-weighted projection

A handle becomes more useful when the projection changes with need.

Let:

$$
N\in\{0,1\}^{k}
$$

select which action dimensions matter.

Then perception can be modeled as:

$$
h_N(x)=\Pi_N(x)
$$

where different needs expose different rows of the same deep state.

This is why the same world renders differently under hunger, threat, curiosity, pain, or repair.


In [ ]:
# Need-weighted handles

def need_weighted_handle(Xbatch: np.ndarray, need_mask: np.ndarray) -> np.ndarray:
    A = action_of_state(Xbatch)
    # Keep needed action bits; fill unneeded bits with nuisance projection.
    nuisance = (Xbatch @ nuisance_projection.T) & 1
    needed = A & need_mask.reshape(1, -1)
    return np.concatenate([needed, nuisance], axis=1).astype(np.uint8)

needs = {
    "all_need": np.ones(ACTION_DIM, dtype=np.uint8),
    "low_need_first_half": np.array([1,1,1,1,0,0,0,0], dtype=np.uint8),
    "low_need_second_half": np.array([0,0,0,0,1,1,1,1], dtype=np.uint8),
    "single_need": np.array([1,0,0,0,0,0,0,0], dtype=np.uint8),
}

need_rows = []
for need_name, mask in needs.items():
    Hn = need_weighted_handle(X, mask)
    rep = collision_report(Hn, need_name)
    rep["need_bits_active"] = int(mask.sum())
    need_rows.append(rep)

need_df = pd.DataFrame(need_rows)
need_df[["handle_type", "need_bits_active", "unique_handles", "conflict_collision_state_rate", "coherent_collision_state_rate"]]


## 8. Clean collapse

The executable lesson:

$$
\boxed{
\text{a handle is not the thing; it is a bounded interface to the thing.}
}
$$

$$
\boxed{
\text{collision is not automatically failure; collision is failure only when action diverges.}
}
$$

$$
\boxed{
\text{residue geometry is the fiber behind the handle.}
}
$$

For SHA work, this gives the corrected framing:

$$
\boxed{
\text{the digest is not ASCII, not hex, and not a noun-value.}
}
$$

It is a fixed-width residue surface produced by a particular binary-field manipulation grammar.

Next SHA rail should use this distinction:

- natural/perceptual hashes preserve action semantics,
- SHA hashes maximize avalanche and destroy semantic locality,
- both are bounded handles,
- inversion must target the residue geometry/fiber, not the GUI row.


In [ ]:
# Export summary tables

collision_df.to_csv("natural_hashing_collision_report.csv", index=False)
summary_gui.to_csv("natural_hashing_gui_rows.csv", index=False)
fiber_df.to_csv("natural_hashing_fiber_geometry.csv", index=False)
sens_df.to_csv("natural_hashing_sensitivity.csv", index=False)
need_df.to_csv("natural_hashing_need_weighted_handles.csv", index=False)

print("Exported:")
print("natural_hashing_collision_report.csv")
print("natural_hashing_gui_rows.csv")
print("natural_hashing_fiber_geometry.csv")
print("natural_hashing_sensitivity.csv")
print("natural_hashing_need_weighted_handles.csv")


# Final Ψ-collapse

$$
\boxed{
\text{Nature is not made of objects. Nature is made of live pressure fields rendering callable handles.}
}
$$

A noun is a stable GUI row.

A perception is a bounded readout.

A memory is a residue handle.

A seed is a generative handle.

A particle is a stable interaction handle.

A SHA digest is an engineered avalanche handle.

The common grammar is:

$$
\boxed{
\text{deep field}
\rightarrow
\text{projection}
\rightarrow
\text{handle}
\rightarrow
\text{action}
}
$$

The next executable bridge back to SHA is:

$$
\boxed{
\text{Phase 1153: controlled-subspace Need descent with handle/fiber language.}
}
$$

There, the candidate message is not a “number guess.” It is a point on a fiber trying to satisfy a fixed-width residue handle.
